In [1]:
import sys
from pathlib import Path
import pandas as pd
# Ajout du dossier source
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from data_loading import load_data
from utils import setup_logger, ensure_directory_exists
from churn_analysis_functions import get_cohort_retention, get_ltv_by_segment


#confi du looger
logger = setup_logger("../reports/logs/analy_load.log")

In [2]:
# Chargement des données
DATA_PATH = Path.cwd().parent / "data" / "processed" / "churn_cleaned.csv"
df = load_data(DATA_PATH)

2026-04-09 06:46:31,223 - INFO - |==> Chargement des données depuis e:\Certifs\Future_Interns\FUTURE_DS_02\data\processed\churn_cleaned.csv


In [3]:
# --- ANALYSE DE LA RÉTENTION (Cohortes d'ancienneté) ---
print("📊 TAUX DE RÉTENTION PAR GROUPE D'ANCIENNETÉ :")
retention_df = get_cohort_retention(df, group_col='tenure_group')
display(retention_df)

📊 TAUX DE RÉTENTION PAR GROUPE D'ANCIENNETÉ :


,tenure_group,total_customers,churned,retention_rate_%
0,0-6 mois,1470,784,46.67
1,1-2 ans,1024,294,71.29
2,2-3 ans,832,180,78.37
3,3-4 ans,762,145,80.97
4,4-5 ans,832,120,85.58
5,5-6 ans,1407,93,93.39
6,6-12 mois,705,253,64.11


**Cohortes Retention:**

Le tableau démontre clairement l'"effet de falaise" (cliff effect). Le taux de rétention des nouveaux clients `(0-6 mois)` est alarmant, la moitié des nouveaux inscrits finissent par partir très vite. À l'inverse, dès qu'un client passe le cap crucial des `2 ans d'abonnement (24+ mois)`, sa fidélité (Rétention) bondit au-dessus de `90%`.

>Conclusion Métier : Le budget marketing (acquisition de nouveaux clients) est gaspillé si notre "Onboarding" et notre service client ne retiennent pas l'usager pendant son premier trimestre.


In [4]:
print("\n💰 VALEUR À VIE DU CLIENT (LTV) SELON LE TYPE DE CONTRAT :")
# Qui rapporte le plus d'argent au final ?
ltv_df = get_ltv_by_segment(df, segment_col='Contract')
display(ltv_df)


💰 VALEUR À VIE DU CLIENT (LTV) SELON LE TYPE DE CONTRAT :


,Contract,arpu,churn_rate,total_clients,LTV_Estimee_$
2,Two year,60.872374,0.028487,1685,2136.87
1,One year,65.079416,0.112772,1472,577.09
0,Month-to-month,66.398490,0.427097,3875,155.46


Insight sur la Valorisation (LTV par type de contrat) :

Cet indicateur est redoutable car il lie la fidélité au chiffre d'affaires. Bien que les contrats mensuels rapportent de l'argent comptant tous les mois (ARPU élevé), leur taux de résiliation (Churn moyen) est tellement fort que leur LTV s'effondre. En revanche, un client avec un `contrat de 2 ans`, même s'il venait à bénéficier d'une petite réduction mensuelle, rapporte une Valeur à Vie `(LTV)` mathématiquement **décuplée**, car sa probabilité de départ approche de zéro.

>Stratégie: Il est mathématiquement rentable de dépenser beaucoup d'argent (offrir Netflix, réductions massives sur les smartphones, gratuité de 3 mois) POURVU que le client accepte de s'engager sur 12 ou 24 mois. L'investissement sera largement amorti.
